# DAY 2 TASK

## IMPORT THE LIBRARY AND LOAD DATASET

In [2]:
import pandas as pd

nav_history = pd.read_csv("../data/raw/02_nav_history.csv")
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")

### check the column and head of nav_history

In [3]:
print(nav_history.columns)
print(nav_history.head())

Index(['amfi_code', 'date', 'nav'], dtype='object')
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


### Clean nav_history.csv

#### Check missing values

In [51]:
print(nav_history.isnull().sum())

amfi_code    0
date         0
nav          0
dtype: int64


#### Convert date to datetime

In [53]:
nav_history["date"] = pd.to_datetime(nav_history["date"])
nav_history["date"]

5750    2022-01-03
5751    2022-01-04
5752    2022-01-05
5753    2022-01-06
5754    2022-01-07
           ...    
45995   2026-05-25
45996   2026-05-26
45997   2026-05-27
45998   2026-05-28
45999   2026-05-29
Name: date, Length: 46000, dtype: datetime64[ns]

#### Sort by fund and date

In [54]:
nav_history = nav_history.sort_values(
    ["amfi_code", "date"]
)
nav_history

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639
...,...,...,...
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873


#### Remove duplicate rows

In [55]:
nav_history = nav_history.drop_duplicates()
nav_history

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639
...,...,...,...
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873


#### Forward fill NAV values

In [57]:
nav_history["nav"] = nav_history.groupby(
    "amfi_code"
)["nav"].ffill()

nav_history["nav"]

5750     520.4608
5751     515.0971
5752     521.7239
5753     515.7880
5754     515.1639
           ...   
45995    292.4810
45996    291.2707
45997    288.8007
45998    280.6873
45999    279.7511
Name: nav, Length: 46000, dtype: float64

#### Validate NAV > 0

In [58]:
invalid_nav = nav_history[
    nav_history["nav"] <= 0
]

print("Invalid NAV Records:")
print(len(invalid_nav))

Invalid NAV Records:
0


#### Save cleaned file

In [59]:
nav_history.to_csv(
    "../data/processed/nav_history_clean.csv",
    index=False
)

### check the column and head of transactions

In [49]:
print(transactions.columns)
print(transactions.head())

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='object')
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              Sip        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              Sip         912   
3   INV003436       2024-01-01     118634              Sip        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra  

# Clean investor_transactions.csv

#### Check transaction types

In [18]:
print(
    transactions["transaction_type"]
    .value_counts()
)

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64


#### Standardize transaction types

In [21]:
transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .str.title()
)
transactions["transaction_type"]

0               Sip
1        Redemption
2               Sip
3               Sip
4           Lumpsum
            ...    
32773       Lumpsum
32774           Sip
32775           Sip
32776           Sip
32777    Redemption
Name: transaction_type, Length: 32778, dtype: object

#### Fix dates

In [23]:
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)
transactions["transaction_date"]

0       2024-01-01
1       2024-01-01
2       2024-01-01
3       2024-01-01
4       2024-01-01
           ...    
32773   2025-05-30
32774   2025-05-30
32775   2025-05-30
32776   2025-05-30
32777   2025-05-30
Name: transaction_date, Length: 32778, dtype: datetime64[ns]

#### Validate amount > 0

In [24]:
invalid_amount = transactions[
    transactions["amount_inr"] <= 0
]

print("Invalid Amount Rows:")
print(len(invalid_amount))

Invalid Amount Rows:
0


#### Check KYC values

In [25]:
print(
    transactions["kyc_status"]
    .value_counts()
)

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


#### Save cleaned file

In [26]:
transactions.to_csv(
    "../data/processed/investor_transactions_clean.csv",
    index=False
)

### check the column and head of performance

In [50]:
print(performance.columns)
print(performance.head())

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='object')
   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct    

# Clean scheme_performance.csv

#### Check numeric columns

In [60]:
numeric_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct"
]

#### Convert to numeric

In [61]:
for col in numeric_cols:
    performance[col] = pd.to_numeric(
        performance[col],
        errors="coerce"
    )
performance

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low
5,100016,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Large Cap,Regular,10.94,14.84,11.32,14.06,0.78,0.97,1.06,1.70,14.0,-17.41,6434,1.55,5,Moderate
6,125497,HDFC Top 100 Fund - Direct Plan - Growth,HDFC Mutual Fund,Large Cap,Direct,11.48,13.38,13.48,12.25,1.13,0.97,0.96,1.45,14.0,-33.50,10611,0.92,4,Moderate
7,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,HDFC Mutual Fund,Mid Cap,Regular,15.43,16.58,17.69,15.63,0.95,0.91,0.87,1.44,19.0,-13.67,23185,1.38,5,High
8,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,Direct,19.98,15.29,15.85,14.39,0.90,1.04,0.80,1.38,19.0,-32.22,18792,0.78,4,High
9,100025,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Short Duration,Regular,6.83,7.37,6.41,5.39,1.98,0.44,1.84,2.79,4.0,-6.01,27953,0.56,3,Low


#### Check missing values

In [62]:
print(
    performance[numeric_cols]
    .isnull()
    .sum()
)

return_1yr_pct       0
return_3yr_pct       0
return_5yr_pct       0
benchmark_3yr_pct    0
alpha                0
beta                 0
sharpe_ratio         0
sortino_ratio        0
std_dev_ann_pct      0
max_drawdown_pct     0
aum_crore            0
expense_ratio_pct    0
dtype: int64


#### Validate Expense Ratio

In [37]:
anomalies = performance[
    (performance["expense_ratio_pct"] < 0.1)
    |
    (performance["expense_ratio_pct"] > 2.5)
]

print(anomalies.shape)

(0, 19)


#### Save cleaned file

In [63]:
performance.to_csv(
    "../data/processed/scheme_performance_clean.csv",
    index=False
)

# INSTALL sqlalchemy LIBRARY

In [64]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Create SQLite Database

In [41]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///../bluestock_mf.db"
)

print("Database Created")

Database Created


#### Load Fund Dimension

In [42]:
fund_master.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

40

#### Load NAV

In [65]:
nav_history.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

46000

#### Load Transactions

In [44]:
transactions.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

32778

#### Load Performance

In [45]:
performance.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

40

#### cleaning all the remaining dataset together

In [48]:
fund_master.to_csv(
    "../data/processed/fund_master_clean.csv",
    index=False
)

pd.read_csv("../data/raw/03_aum_by_fund_house.csv").to_csv(
    "../data/processed/aum_by_fund_house_clean.csv",
    index=False
)

pd.read_csv("../data/raw/04_monthly_sip_inflows.csv").to_csv(
    "../data/processed/monthly_sip_inflows_clean.csv",
    index=False
)

pd.read_csv("../data/raw/05_category_inflows.csv").to_csv(
    "../data/processed/category_inflows_clean.csv",
    index=False
)

pd.read_csv("../data/raw/06_industry_folio_count.csv").to_csv(
    "../data/processed/industry_folio_count_clean.csv",
    index=False
)

pd.read_csv("../data/raw/09_portfolio_holdings.csv").to_csv(
    "../data/processed/portfolio_holdings_clean.csv",
    index=False
)

pd.read_csv("../data/raw/10_benchmark_indices.csv").to_csv(
    "../data/processed/benchmark_indices_clean.csv",
    index=False
)

print("All processed files saved successfully.")

All processed files saved successfully.
